# 01 — Training scaffold

**SCAFFOLD — NOT RUN.** Prototype instructions for Colab. P1 training data are synthetic; this notebook does not establish real-world performance. LogisticRegression is the baseline; Random Forest, XGBoost, and LightGBM are optional where native imports work. Spatial stacking needs group-safe out-of-fold probabilities and is intentionally left as a TODO. Replace synthetic data only with licensed, provenance-tracked observations. Never use its output for warnings or evacuation decisions. Not a replacement for official IMD/NDMA/GSI warnings.

In [ ]:
# In Colab, clone the project and install requirements before running cells.
# Do not upload restricted, personal, or login-only data.
from backend.app.ml.susceptibility import _model_factories
from backend.app.ml.synthetic_data import FEATURE_COLUMNS, generate_nilgiris_pilot
from sklearn.model_selection import GroupKFold

pilot = generate_nilgiris_pilot(grid_size=40, seed=42)
features = pilot.cells.loc[:, FEATURE_COLUMNS]
target = pilot.cells["landslide_label"].to_numpy()
groups = pilot.cells["spatial_block"].to_numpy()
factories = _model_factories(seed=42)  # Falls back to LogisticRegression if native tree imports fail.
splitter = GroupKFold(n_splits=min(5, len(set(groups))))

# Each fold must keep spatial blocks disjoint. This is a SIMULATED baseline.
fold_models = []
for train_index, validation_index in splitter.split(features, target, groups):
    fold_models.append({
        name: factory().fit(features.iloc[train_index], target[train_index])
        for name, factory in factories.items()
    })

# TODO: add a leakage-safe spatial stacking implementation before comparing a stack.
# Do not use ordinary shuffled stacking folds for spatially autocorrelated data.